# Module 04: Data Modeling

**Program:** Quintrix Jr. Data Engineer Training
**Duration:** 2 hours
**Instructor:** Sunil Mogadati

---

**Today's Agenda:**

| # | Topic | Time |
|---|-------|------|
| 1 | Why Data Modeling? (OLTP → OLAP) | 10 min |
| 2 | Star Schema — The Gold Standard | 15 min |
| 3 | Building Dimension Tables | 20 min |
| 4 | Slowly Changing Dimensions (SCD) | 20 min |
| 5 | Building the Fact Table | 15 min |
| 6 | Star Schema Queries | 10 min |
| 7 | Partitioning & Bucketing | 10 min |
| 8 | Challenge Lab | 10 min |
| 9 | Key Takeaways | 5 min |
| 10 | Homework & Preview | 5 min |

---

## 1. Why Data Modeling? (OLTP → OLAP)

**One-line answer:** Data modeling transforms messy operational data into clean, queryable structures optimized for analysis.


> **Analogy:** OLTP is the cash register — it records one sale at a time, fast and accurate. OLAP is the end-of-month report — it reads every transaction to find patterns and totals. Our call center database (M02–M03) is OLTP. Today we transform it into OLAP.

| | OLTP | OLAP |
|---|------|------|
| **Purpose** | Run the business (transactions) | Analyze the business (reporting) |
| **Schema** | Normalized (3NF — Third Normal Form, a database design with minimal data duplication, covered in M00 Section 16a) — minimal redundancy | Denormalized (star/snowflake — snowflake is a variation of star schema where dimensions have sub-dimensions; we focus on star schema in this module) — optimized for reads |
| **Queries** | Short, targeted (INSERT/UPDATE one row) | Wide, aggregating (SUM/AVG across millions) |
| **Users** | Applications, APIs | Analysts, dashboards, ML models |
| **Example** | `INSERT INTO orders ...` | `SELECT campaign, SUM(revenue) GROUP BY ...` |

Our call center database (M02–M03) is **OLTP** — it stores individual calls, orders, and payments as they happen. Today we transform it into **OLAP** — a structure designed for fast, intuitive analytics.

> **Medallion architecture connection:** Bronze (raw ingestion) → Silver (cleaned/validated) → **Gold (modeled for analytics) = star schema**. Today we build the Gold layer.

In [ ]:
import sqlite3
from datetime import date, timedelta

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# --- Create OLTP tables (same schema as M02/M03) ---
cursor.executescript('''
CREATE TABLE clients (
    client_id INTEGER PRIMARY KEY,
    client_name TEXT NOT NULL,
    industry TEXT
);

CREATE TABLE sources (
    dnis TEXT PRIMARY KEY,
    campaign_name TEXT NOT NULL,
    client_name TEXT NOT NULL,
    channel TEXT NOT NULL
);

CREATE TABLE calls (
    call_id TEXT PRIMARY KEY,
    dnis TEXT NOT NULL,
    caller_ani TEXT,
    call_started_at TEXT NOT NULL,
    call_ended_at TEXT,
    disposition TEXT,
    sentiment TEXT,
    channel TEXT NOT NULL
);

CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    call_id TEXT,
    sku TEXT NOT NULL,
    subtotal REAL,
    tax REAL,
    shipping REAL,
    total REAL,
    FOREIGN KEY (call_id) REFERENCES calls(call_id)
);

CREATE TABLE payments (
    transaction_id TEXT PRIMARY KEY,
    order_id TEXT,
    auth_code TEXT,
    amount REAL,
    status TEXT,
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);

CREATE TABLE products (
    sku TEXT PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    unit_price REAL NOT NULL
);
''')

# --- Insert OLTP data ---
cursor.executemany("INSERT INTO clients VALUES (?,?,?)", [
    (1, "Acme Corp", "Retail"),
    (2, "Pinnacle Health", "Healthcare"),
    (3, "Summit Financial", "Financial Services"),
])

cursor.executemany("INSERT INTO sources VALUES (?,?,?,?)", [
    ("8005551234", "Acme Spring Sale", "Acme Corp", "live_agent"),
    ("8005551235", "Acme Rewards Program", "Acme Corp", "va"),
    ("8005552345", "Pinnacle Wellness Plan", "Pinnacle Health", "live_agent"),
    ("8005552346", "Pinnacle Rx Refill", "Pinnacle Health", "va"),
    ("8005553456", "Summit Auto Loan", "Summit Financial", "live_agent"),
    ("8005553457", "Summit Credit Card", "Summit Financial", "va"),
])

cursor.executemany("INSERT INTO calls VALUES (?,?,?,?,?,?,?,?)", [
    ("call_001", "8005551234", "3135559876", "2026-03-10T13:00:00Z", "2026-03-10T13:07:45Z", "completed", "positive", "live_agent"),
    ("call_002", "8005551234", "2485555678", "2026-03-10T13:30:00Z", "2026-03-10T13:32:15Z", "dropped", "negative", "live_agent"),
    ("call_003", "8005552345", "7345551234", "2026-03-10T14:00:00Z", "2026-03-10T14:12:30Z", "completed", "neutral", "live_agent"),
    ("call_004", "8005552346", "3135554321", "2026-03-10T14:30:00Z", "2026-03-10T14:35:00Z", "transferred", "neutral", "va"),
    ("call_005", "8005553456", "2485559999", "2026-03-10T15:00:00Z", "2026-03-10T15:08:20Z", "completed", "positive", "live_agent"),
    ("call_006", "8005551235", "5865551111", "2026-03-10T15:30:00Z", "2026-03-10T15:31:10Z", "dropped", "negative", "va"),
    ("call_007", "8005553457", "3135558888", "2026-03-10T16:00:00Z", "2026-03-10T16:06:45Z", "completed", "positive", "va"),
    ("call_008", "8005551234", "7345557777", "2026-03-10T16:30:00Z", "2026-03-10T16:42:00Z", "completed", "neutral", "live_agent"),
    ("call_009", "8005552345", "2485556666", "2026-03-10T17:00:00Z", "2026-03-10T17:01:30Z", "dropped", "negative", "live_agent"),
    ("call_010", "8005553456", "3135553333", "2026-03-10T17:30:00Z", "2026-03-10T17:38:15Z", "completed", "positive", "live_agent"),
    ("call_011", "8005551234", "5865552222", "2026-03-10T18:00:00Z", "2026-03-10T18:09:30Z", "completed", "neutral", "live_agent"),
    ("call_012", "8005552346", "7345554444", "2026-03-10T18:30:00Z", "2026-03-10T18:33:00Z", "transferred", "neutral", "va"),
    ("call_013", "8005551235", "3135551111", "2026-03-10T19:00:00Z", "2026-03-10T19:05:45Z", "completed", "positive", "va"),
    ("call_014", "8005553457", "2485553333", "2026-03-10T19:30:00Z", "2026-03-10T19:31:00Z", "dropped", "negative", "va"),
    ("call_015", "8005551234", "5865554444", "2026-03-10T20:00:00Z", "2026-03-10T20:11:20Z", "completed", "positive", "live_agent"),
    ("call_016", "8005552345", "7345558888", "2026-03-10T20:30:00Z", "2026-03-10T20:36:00Z", "completed", "neutral", "live_agent"),
    ("call_017", "8005553456", "3135559999", "2026-03-10T21:00:00Z", "2026-03-10T21:02:00Z", "dropped", "negative", "live_agent"),
    ("call_018", "8005551235", "2485551111", "2026-03-10T21:30:00Z", "2026-03-10T21:40:15Z", "completed", "neutral", "va"),
    ("call_019", "8005552346", "5865557777", "2026-03-10T22:00:00Z", "2026-03-10T22:04:30Z", "transferred", "neutral", "va"),
    ("call_020", "8005553457", "7345552222", "2026-03-10T22:30:00Z", "2026-03-10T22:37:45Z", "completed", "positive", "va"),
    ("call_021", "8005551234", "3135556666", "2026-03-11T00:00:00Z", "2026-03-11T00:09:30Z", "completed", "neutral", "live_agent"),
    ("call_022", "8005552345", "2485558888", "2026-03-11T00:30:00Z", "2026-03-11T00:31:45Z", "dropped", "negative", "live_agent"),
    ("call_023", "8005553456", "5865553333", "2026-03-11T01:00:00Z", "2026-03-11T01:06:00Z", "completed", "positive", "live_agent"),
    ("call_024", "8005551235", "7345559999", "2026-03-11T01:30:00Z", "2026-03-11T01:38:20Z", "completed", "positive", "va"),
    ("call_025", "8005553457", "3135552222", "2026-03-11T02:00:00Z", "2026-03-11T02:03:15Z", "transferred", "neutral", "va"),
    ("call_026", "8005551234", "2485554444", "2026-03-11T13:00:00Z", "2026-03-11T13:05:30Z", "completed", "positive", "live_agent"),
    ("call_027", "8005552345", "5865556666", "2026-03-11T13:30:00Z", "2026-03-11T13:32:00Z", "dropped", "negative", "live_agent"),
    ("call_028", "8005553456", "7345553333", "2026-03-11T14:00:00Z", "2026-03-11T14:10:45Z", "completed", "neutral", "live_agent"),
    ("call_029", "8005551235", "3135557777", "2026-03-11T14:30:00Z", "2026-03-11T14:34:00Z", "transferred", "neutral", "va"),
    ("call_030", "8005552346", "2485552222", "2026-03-11T15:00:00Z", "2026-03-11T15:08:30Z", "completed", "positive", "va"),
    ("call_031", "8005553457", "5865559999", "2026-03-11T15:30:00Z", "2026-03-11T15:31:20Z", "dropped", "negative", "va"),
    ("call_032", "8005551234", "7345556666", "2026-03-11T16:00:00Z", "2026-03-11T16:07:00Z", "completed", "neutral", "live_agent"),
    ("call_033", "8005552345", "3135554444", "2026-03-11T16:30:00Z", "2026-03-11T16:41:30Z", "completed", "positive", "live_agent"),
    ("call_034", "8005553456", "2485557777", "2026-03-11T17:00:00Z", "2026-03-11T17:01:45Z", "dropped", "negative", "live_agent"),
    ("call_035", "8005551235", "5865558888", "2026-03-11T17:30:00Z", "2026-03-11T17:36:15Z", "completed", "positive", "va"),
    ("call_036", "8005552346", "7345551111", "2026-03-11T18:00:00Z", "2026-03-11T18:03:30Z", "transferred", "neutral", "va"),
    ("call_037", "8005553457", "3135558888", "2026-03-11T18:30:00Z", "2026-03-11T18:39:45Z", "completed", "neutral", "va"),
    ("call_038", "8005551234", "2485559876", "2026-03-11T19:00:00Z", "2026-03-11T19:06:20Z", "completed", "positive", "live_agent"),
    ("call_039", "8005552345", "5865554321", "2026-03-11T19:30:00Z", "2026-03-11T19:32:00Z", "dropped", "negative", "live_agent"),
    ("call_040", "8005553456", "7345559876", "2026-03-11T20:00:00Z", "2026-03-11T20:07:30Z", "completed", "positive", "live_agent"),
])

cursor.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?)", [
    ("ord_001", "call_001", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_002", "call_003", "SKU-PH-2001", 99.99, 8.00, 11.99, 119.98),
    ("ord_003", "call_005", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_004", "call_008", "SKU-AC-1002", 139.97, 11.20, 8.80, 159.97),
    ("ord_005", "call_010", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_006", "call_011", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_007", "call_015", "SKU-AC-1003", 89.99, 7.20, 5.80, 102.99),
    ("ord_008", "call_016", "SKU-PH-2002", 59.95, 4.80, 5.20, 69.95),
    ("ord_009", "call_020", "SKU-SF-3002", 149.99, 12.00, 7.00, 168.99),
    ("ord_010", "call_021", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_011", "call_023", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_012", "call_026", "SKU-AC-1002", 139.97, 11.20, 8.80, 159.97),
    ("ord_013", "call_030", "SKU-PH-2001", 99.99, 8.00, 11.99, 119.98),
    ("ord_014", "call_033", "SKU-PH-2002", 59.95, 4.80, 5.20, 69.95),
    ("ord_015", "call_040", "SKU-SF-3002", 149.99, 12.00, 7.00, 168.99),
])

cursor.executemany("INSERT INTO payments VALUES (?,?,?,?,?)", [
    ("txn_001", "ord_001", "AUTH-7821", 79.99, "approved"),
    ("txn_002", "ord_002", "AUTH-7822", 119.98, "approved"),
    ("txn_003", "ord_003", "AUTH-7823", 49.95, "approved"),
    ("txn_004", "ord_004", "AUTH-7824", 159.97, "approved"),
    ("txn_005", "ord_005", "AUTH-7825", 49.95, "approved"),
    ("txn_006", "ord_006", None, 79.99, "declined"),
    ("txn_007", "ord_007", "AUTH-7827", 102.99, "approved"),
    ("txn_008", "ord_008", "AUTH-7828", 69.95, "approved"),
    ("txn_009", "ord_009", "AUTH-7829", 168.99, "approved"),
    ("txn_010", "ord_010", "AUTH-7830", 79.99, "approved"),
    ("txn_011", "ord_011", "AUTH-7831", 49.95, "approved"),
    ("txn_012", "ord_012", None, 159.97, "declined"),
    ("txn_013", "ord_013", "AUTH-7833", 119.98, "approved"),
    ("txn_014", "ord_014", "AUTH-7834", 69.95, "approved"),
    ("txn_015", "ord_015", "AUTH-7835", 168.99, "approved"),
])

# --- NEW: Products reference table ---
cursor.executemany("INSERT INTO products VALUES (?,?,?,?)", [
    ("SKU-AC-1001", "Acme Widget Basic", "Widgets", 69.99),
    ("SKU-AC-1002", "Acme Widget Pro Bundle", "Widgets", 139.97),
    ("SKU-AC-1003", "Acme Premium Widget", "Widgets", 89.99),
    ("SKU-PH-2001", "Pinnacle Wellness Kit", "Health & Wellness", 99.99),
    ("SKU-PH-2002", "Pinnacle Supplement Pack", "Health & Wellness", 59.95),
    ("SKU-SF-3001", "Summit Loan Protection Plan", "Financial Products", 39.95),
    ("SKU-SF-3002", "Summit Premium Coverage", "Financial Products", 149.99),
])

conn.commit()

# --- Verify ---
print("=== OLTP Database Rebuilt ===")
for table in ["calls", "orders", "payments", "sources", "clients", "products"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table + ':':<12} {count:>3} rows")
print()
print("This is our OLTP system. Now let's model it for analytics.")

---

## 2. Star Schema — The Gold Standard

**One-line answer:** A star schema has one fact table (measurements) surrounded by dimension tables (context).
> **Analogy:** Think of a star schema like a hub-and-spoke wheel: `fact_calls` is the hub at the center, and each dimension table (date, time, campaign, product, customer) is a spoke radiating outward. Every measurement (a call, a sale) sits at the hub and connects to context through the spokes.


```
                     dim_date
                        |
  dim_campaign ── fact_calls ── dim_product
                        |
  dim_customer ─────────┘── dim_time
```

| | Fact Table | Dimension Table |
|---|-----------|----------------|
| **Contains** | Measures (numbers) | Context (who, what, when, where) |
| **Rows** | One per event/transaction | One per entity (or version, for SCD) |
| **Changes** | Append-only (new events) | Slowly changes (updates, corrections) |
| **Keys** | Foreign keys to dimensions | Surrogate keys (integers) |
| **Example** | `fact_calls`: duration, revenue, is_converted | `dim_date`: day_name, is_weekend |

**Grain statement:** One row in `fact_calls` = **one phone call**. This is the most important decision in star schema design — everything follows from it.

Our 5 dimensions:
1. **dim_date** — when (day, day of week, weekend flag)
2. **dim_time** — when (hour, AM/PM, time period)
3. **dim_campaign** — which campaign/client (denormalized from sources + clients)
4. **dim_product** — what was sold (from the new products table)
5. **dim_customer** — who called (caller phone number, city, state)

### Surrogate Keys

**Why surrogate keys?** Natural keys (phone numbers, SKUs, campaign names) are strings — slow to join, can change, have gaps. Surrogate keys are auto-incrementing integers: fast, stable, compact.

| Dimension | Natural Key | Surrogate Key |
|-----------|-------------|---------------|
| dim_date | `'2026-03-10'` (TEXT) | `20260310` (INTEGER) |
| dim_time | `'08:00'` (TEXT) | `8` (INTEGER) |
| dim_campaign | `'8005551234'` (TEXT) | `1` (INTEGER) |
| dim_product | `'SKU-AC-1001'` (TEXT) | `1` (INTEGER) |
| dim_customer | `'3135559876'` (TEXT) | `1` (INTEGER) |

**Default dimension row (key = 0):** Every dimension gets a "Unknown/N/A" row with key 0. A call with no order gets `product_key = 0` (Unknown Product) instead of NULL. This eliminates NULLs in the fact table — every foreign key always points to a valid dimension row.

> **Rule:** No NULLs in fact table foreign keys. Use default dimension rows instead.

---

## 3. Building Dimension Tables

**One-line answer:** Dimension tables provide the "who, what, when, where" context that makes fact table numbers meaningful.

Each dimension table below serves a specific purpose:
- **dim_date** — pre-built calendar lookup so analysts filter by day name, weekend, month without parsing timestamps
- **dim_time** — hourly periods (Morning, Afternoon, Evening, Night) for call volume analysis
- **dim_campaign** — campaign details and tier classification
- **dim_customer** — customer info with SCD Type 2 history (Section 4)
- **dim_product** — product catalog for order analysis

In [ ]:
# --- Build dim_date: one row per calendar day ---
cursor.execute('''
    CREATE TABLE dim_date (
        date_key INTEGER PRIMARY KEY,
        full_date TEXT NOT NULL,
        year INTEGER,
        month INTEGER,
        day INTEGER,
        day_of_week INTEGER,
        day_name TEXT,
        is_weekend INTEGER
    )
''')

day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
start = date(2026, 3, 1)

for i in range(31):
    d = start + timedelta(days=i)
    cursor.execute(
        "INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?)",
        (int(d.strftime('%Y%m%d')), d.isoformat(), d.year, d.month, d.day,
         d.isoweekday(), day_names[d.weekday()], 1 if d.weekday() >= 5 else 0)
    )

conn.commit()

# --- Display ---
print("=== dim_date (March 2026) ===")
print(f"{'date_key':>10}  {'full_date':<12}  {'year':>4}  {'month':>5}  {'day':>3}  {'dow':>3}  {'day_name':<11}  {'weekend':>7}")
print("-" * 72)
rows = cursor.execute("SELECT * FROM dim_date LIMIT 7").fetchall()
for r in rows:
    wknd = "Yes" if r[7] else "No"
    print(f"{r[0]:>10}  {r[1]:<12}  {r[2]:>4}  {r[3]:>5}  {r[4]:>3}  {r[5]:>3}  {r[6]:<11}  {wknd:>7}")

total = cursor.execute("SELECT COUNT(*) FROM dim_date").fetchone()[0]
print(f"\n{total} rows total — one per calendar day.")
print("Analysts filter by day_name, is_weekend — they never parse timestamps.")

In [ ]:
# --- Build dim_time: one row per hour (0-23) ---
cursor.execute('''
    CREATE TABLE dim_time (
        time_key INTEGER PRIMARY KEY,
        hour_24 INTEGER NOT NULL,
        hour_12 INTEGER NOT NULL,
        am_pm TEXT NOT NULL,
        time_period TEXT NOT NULL
    )
''')

for h in range(24):
    hour_12 = h % 12 if h % 12 != 0 else 12
    am_pm = 'AM' if h < 12 else 'PM'
    if h < 6:
        period = 'Night'
    elif h < 12:
        period = 'Morning'
    elif h < 18:
        period = 'Afternoon'
    else:
        period = 'Evening'
    cursor.execute("INSERT INTO dim_time VALUES (?,?,?,?,?)", (h, h, hour_12, am_pm, period))

conn.commit()

# --- Display all 24 rows ---
print("=== dim_time (24 hours) ===")
print(f"{'time_key':>8}  {'hour_24':>7}  {'hour_12':>7}  {'am_pm':<5}  {'time_period':<10}")
print("-" * 44)
for r in cursor.execute("SELECT * FROM dim_time"):
    print(f"{r[0]:>8}  {r[1]:>7}  {r[2]:>7}  {r[3]:<5}  {r[4]:<10}")

print("\ntime_period lets analysts ask 'How do evening calls compare to morning?'")
print("without knowing the hour boundaries.")

In [ ]:
# --- Build dim_campaign: denormalize sources + clients ---
cursor.execute('''
    CREATE TABLE dim_campaign (
        campaign_key INTEGER PRIMARY KEY AUTOINCREMENT,  -- AUTOINCREMENT: database generates the next integer automatically — no manual ID management
        dnis TEXT NOT NULL,
        campaign_name TEXT NOT NULL,
        client_name TEXT NOT NULL,
        industry TEXT,
        channel TEXT NOT NULL
    )
''')

# Flatten the JOIN — one table instead of two
cursor.execute('''
    INSERT INTO dim_campaign (dnis, campaign_name, client_name, industry, channel)
    SELECT s.dnis, s.campaign_name, s.client_name, cl.industry, s.channel
    FROM sources s
    JOIN clients cl ON s.client_name = cl.client_name
    ORDER BY s.dnis
''')

conn.commit()

# --- Display ---
print("=== dim_campaign ===")
print(f"{'key':>3}  {'dnis':<12}  {'campaign_name':<24}  {'client_name':<18}  {'industry':<20}  {'channel':<10}")
print("-" * 92)
for r in cursor.execute("SELECT * FROM dim_campaign"):
    print(f"{r[0]:>3}  {r[1]:<12}  {r[2]:<24}  {r[3]:<18}  {r[4]:<20}  {r[5]:<10}")

print("\nIn OLTP, sources and clients are separate tables (normalized).")
print("In the star schema, we denormalize — one table, no JOINs needed.")

In [ ]:
# --- Build dim_product: products + default row ---
cursor.execute('''
    CREATE TABLE dim_product (
        product_key INTEGER PRIMARY KEY,
        sku TEXT,
        product_name TEXT NOT NULL,
        category TEXT NOT NULL,
        unit_price REAL
    )
''')

# Row 0 = default for calls with no order
products_dim = [
    (0, "N/A", "Unknown Product", "Unknown", 0.00),
    (1, "SKU-AC-1001", "Acme Widget Basic", "Widgets", 69.99),
    (2, "SKU-AC-1002", "Acme Widget Pro Bundle", "Widgets", 139.97),
    (3, "SKU-AC-1003", "Acme Premium Widget", "Widgets", 89.99),
    (4, "SKU-PH-2001", "Pinnacle Wellness Kit", "Health & Wellness", 99.99),
    (5, "SKU-PH-2002", "Pinnacle Supplement Pack", "Health & Wellness", 59.95),
    (6, "SKU-SF-3001", "Summit Loan Protection Plan", "Financial Products", 39.95),
    (7, "SKU-SF-3002", "Summit Premium Coverage", "Financial Products", 149.99),
]

cursor.executemany("INSERT INTO dim_product VALUES (?,?,?,?,?)", products_dim)
conn.commit()

# --- Display ---
print("=== dim_product ===")
print(f"{'key':>3}  {'sku':<14}  {'product_name':<30}  {'category':<20}  {'price':>8}")
print("-" * 80)
for r in cursor.execute("SELECT * FROM dim_product"):
    print(f"{r[0]:>3}  {r[1]:<14}  {r[2]:<30}  {r[3]:<20}  ${r[4]:>7.2f}")

print("\n8 rows (7 products + 1 default).")
print("The default row (key=0) avoids NULLs in fact_calls. Every call has a product_key.")

---

## 4. Slowly Changing Dimensions (SCD)

**One-line answer:** SCD Type 2 preserves history by adding a new row when an attribute changes, instead of overwriting.
> **Analogy:** Think of SCD Type 2 like passport history. When you renew your passport, the government doesn’t erase the old one — they keep both on file with date ranges. If a crime happened in 2019, they check which passport was valid then, not the one you hold today. That’s exactly how Type 2 works: every version of a customer record is kept, with `effective_date` and `end_date` marking when each version was active.


| Type | Strategy | History? | Example |
|------|----------|----------|---------|
| **Type 0** | Never update | N/A | Date of birth — immutable |
| **Type 1** | Overwrite the value | Lost | Fix a typo in a name — old value gone |
| **Type 2** | Add new row with date range | Preserved | Customer moves cities — both addresses kept |
| **Type 3** | Add column for previous value | Limited | `current_city` + `previous_city` — only 1 version back |

> **Type 2 is the most important for Data Engineers.** It answers the question: *"What was true at the time of the transaction?"*

### How SCD Type 2 Works

Each row gets three extra columns:
- `effective_date` — when this version became active
- `end_date` — when this version was replaced (9999-12-31 = still active)
> **Why `9999-12-31`?** The date `9999-12-31` is an industry convention meaning “still active” — it’s easier to query than NULL because you can write `WHERE end_date > '2024-01-15'` without special NULL handling.

- `is_current` — 1 for the active row, 0 for historical rows

When a customer moves from Detroit to Chicago on March 11:
1. **Close** the old row: set `end_date = '2026-03-10'`, `is_current = 0`
2. **Insert** a new row: Chicago, `effective_date = '2026-03-11'`, `end_date = '9999-12-31'`, `is_current = 1`

In [ ]:
# --- Build dim_customer with SCD Type 2 ---
cursor.execute('''
    CREATE TABLE dim_customer (
        customer_key INTEGER PRIMARY KEY,
        caller_ani TEXT NOT NULL,
        city TEXT,
        state TEXT,
        effective_date TEXT NOT NULL,
        end_date TEXT NOT NULL,
        is_current INTEGER NOT NULL DEFAULT 1
    )
''')

# Area code to city/state mapping (Michigan area codes)
area_city = {
    '313': ('Detroit', 'MI'),
    '248': ('Southfield', 'MI'),
    '734': ('Ann Arbor', 'MI'),
    '586': ('Warren', 'MI'),
}

# Extract unique callers and assign surrogate keys
callers = cursor.execute(
    "SELECT DISTINCT caller_ani FROM calls ORDER BY caller_ani"
).fetchall()

for i, (ani,) in enumerate(callers, start=1):
    area_code = ani[:3]
    city, state = area_city.get(area_code, ('Unknown', 'XX'))
    cursor.execute(
        "INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?)",
        (i, ani, city, state, '2026-03-01', '9999-12-31', 1)
    )

# --- SCD Type 2 Demo ---
# Customer 3135559876 moved from Detroit, MI to Chicago, IL on March 11

# Step 1: Close the current row
cursor.execute('''
    UPDATE dim_customer
    SET end_date = '2026-03-10', is_current = 0
    WHERE caller_ani = '3135559876' AND is_current = 1
''')

# Step 2: Insert new row with updated city
next_key = len(callers) + 1
cursor.execute(
    "INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?)",
    (next_key, '3135559876', 'Chicago', 'IL', '2026-03-11', '9999-12-31', 1)
)

conn.commit()

# --- Display ---
total = cursor.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]
print(f"=== dim_customer (SCD Type 2) — {total} rows ===")
print()
print(f"{'key':>4}  {'caller_ani':<12}  {'city':<12}  {'st':<3}  {'effective':<12}  {'end_date':<12}  {'cur':>3}")
print("-" * 68)
rows = cursor.execute("SELECT * FROM dim_customer LIMIT 5").fetchall()
for r in rows:
    print(f"{r[0]:>4}  {r[1]:<12}  {r[2]:<12}  {r[3]:<3}  {r[4]:<12}  {r[5]:<12}  {r[6]:>3}")
print("...")

# Show the SCD2 rows
print()
print("=== SCD Type 2 Demo: Caller 3135559876 ===")
scd_rows = cursor.execute(
    "SELECT * FROM dim_customer WHERE caller_ani = '3135559876' ORDER BY effective_date"
).fetchall()
for r in scd_rows:
    print(f"{r[0]:>4}  {r[1]:<12}  {r[2]:<12}  {r[3]:<3}  {r[4]:<12}  {r[5]:<12}  {r[6]:>3}")
print()
print("Same person, two rows. The effective_date tells you which row was active when.")

In [ ]:
# --- Point-in-Time Lookup: SCD Type 2 ---

# Where was caller 3135559876 on March 10?
print("=== Where was 3135559876 on March 10? ===")
row = cursor.execute('''
    SELECT customer_key, caller_ani, city, state, effective_date, end_date
    FROM dim_customer
    WHERE caller_ani = '3135559876'
      AND '2026-03-10' BETWEEN effective_date AND end_date
''').fetchone()
print(f"  Key: {row[0]}, City: {row[2]}, State: {row[3]}")
print(f"  Active: {row[4]} to {row[5]}")

# Where was caller 3135559876 on March 12?
print()
print("=== Where was 3135559876 on March 12? ===")
row = cursor.execute('''
    SELECT customer_key, caller_ani, city, state, effective_date, end_date
    FROM dim_customer
    WHERE caller_ani = '3135559876'
      AND '2026-03-12' BETWEEN effective_date AND end_date
''').fetchone()
print(f"  Key: {row[0]}, City: {row[2]}, State: {row[3]}")
print(f"  Active: {row[4]} to {row[5]}")

print()
print("Same person, different answer depending on WHEN you ask.")
print("This is how you answer 'Where did the customer live WHEN they called?'")
print("Not where they live NOW.")

---

## 5. Building the Fact Table

**Grain:** One row in `fact_calls` = one phone call.

**Foreign keys** (pointers to dimensions):
- `date_key` → dim_date
- `time_key` → dim_time
- `campaign_key` → dim_campaign
- `product_key` → dim_product (0 if no order)
- `customer_key` → dim_customer (SCD Type 2 date-range lookup)

**Measures** (numbers for aggregation):
- `duration_seconds`, `duration_minutes` — call length
- `order_amount` — revenue (0 if no order)
- `is_converted` — 1 if the call generated an order, 0 otherwise
- `is_paid` — 1 if payment was approved, 0 otherwise

**Degenerate dimensions** (a dimension value stored directly in the fact table instead of in a separate dimension table — like `call_id`, which is unique per row and doesn’t need a lookup):
- `disposition` — completed, dropped, transferred
- `sentiment` — positive, neutral, negative

> **Rule:** The fact table has numbers (measures) and keys (pointers). No descriptive text — except degenerate dimensions that don't warrant their own table.

In [ ]:
# --- Build fact_calls ---
cursor.execute('''
    CREATE TABLE fact_calls (
        call_id TEXT PRIMARY KEY,
        date_key INTEGER NOT NULL,
        time_key INTEGER NOT NULL,
        campaign_key INTEGER NOT NULL,
        product_key INTEGER NOT NULL,
        customer_key INTEGER NOT NULL,
        duration_seconds INTEGER,
        duration_minutes REAL,
        order_amount REAL DEFAULT 0,
        disposition TEXT,
        sentiment TEXT,
        is_converted INTEGER DEFAULT 0,
        is_paid INTEGER DEFAULT 0,
        FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
        FOREIGN KEY (time_key) REFERENCES dim_time(time_key),
        FOREIGN KEY (campaign_key) REFERENCES dim_campaign(campaign_key),
        FOREIGN KEY (product_key) REFERENCES dim_product(product_key),
        FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key)
    )
''')

# Transform OLTP → fact table with dimension key lookups
# KEY SQL CONCEPTS in this query:
# - julianday() converts timestamps to decimal days; subtract two = elapsed time
#   * 86400 converts days to seconds for duration
# - COALESCE(..., 0) returns 0 when no matching product/order exists (instead of NULL)
# - '-5 hours': converts UTC to EST (fixes the UTC bug from M02)
# - The SCD Type 2 JOIN: match the customer version that was active at call time
cursor.execute('''
    INSERT INTO fact_calls
    SELECT
        c.call_id,
        dd.date_key,
        dt.time_key,
        dcamp.campaign_key,
        -- COALESCE(a, b) = returns a if not NULL, otherwise returns b. Prevents NULL contamination.
        COALESCE(dprod.product_key, 0) AS product_key,
        dcust.customer_key,
        -- julianday() = SQLite's way to do date math. Returns days as a decimal number.
        CAST((julianday(c.call_ended_at) - julianday(c.call_started_at)) * 86400 AS INTEGER),
        ROUND((julianday(c.call_ended_at) - julianday(c.call_started_at)) * 1440, 2),
        COALESCE(o.total, 0.0),
        c.disposition,
        c.sentiment,
        CASE WHEN o.order_id IS NOT NULL THEN 1 ELSE 0 END,
        CASE WHEN pay.status = 'approved' THEN 1 ELSE 0 END
    FROM calls c
    JOIN dim_date dd
        ON dd.full_date = DATE(c.call_started_at, '-5 hours')
    JOIN dim_time dt
        ON dt.time_key = CAST(strftime('%H', c.call_started_at, '-5 hours') AS INTEGER)
    JOIN dim_campaign dcamp
        ON dcamp.dnis = c.dnis
    JOIN dim_customer dcust
        ON dcust.caller_ani = c.caller_ani
        AND DATE(c.call_started_at, '-5 hours') BETWEEN dcust.effective_date AND dcust.end_date
    LEFT JOIN orders o
        ON o.call_id = c.call_id
    LEFT JOIN payments pay
        ON pay.order_id = o.order_id
    LEFT JOIN dim_product dprod
        ON dprod.sku = o.sku
''')

conn.commit()

# --- Display ---
total = cursor.execute("SELECT COUNT(*) FROM fact_calls").fetchone()[0]
print(f"=== fact_calls — {total} rows ===")
print()
print(f"{'call_id':<10} {'date_key':>8} {'time':>4} {'camp':>4} {'prod':>4} {'cust':>4}  {'dur_s':>5}  {'amount':>8}  {'disp':<11} {'conv':>4} {'paid':>4}")
print("-" * 88)
rows = cursor.execute("SELECT * FROM fact_calls LIMIT 10").fetchall()
for r in rows:
    print(f"{r[0]:<10} {r[1]:>8} {r[2]:>4} {r[3]:>4} {r[4]:>4} {r[5]:>4}  {r[6]:>5}  ${r[8]:>7.2f}  {r[9]:<11} {r[11]:>4} {r[12]:>4}")
print("... (showing first 10)")

# Verify counts
converted = cursor.execute("SELECT SUM(is_converted) FROM fact_calls").fetchone()[0]
paid = cursor.execute("SELECT SUM(is_paid) FROM fact_calls").fetchone()[0]
revenue = cursor.execute("SELECT SUM(order_amount) FROM fact_calls").fetchone()[0]
print(f"\n40 OLTP calls -> {total} fact rows. {converted} conversions, {paid} paid, ${revenue:,.2f} total revenue.")
print("Each row points to 5 dimensions via integer keys.")

---

## 6. Star Schema Queries

**Now the payoff.** Queries on a star schema are simpler, faster, and more intuitive than OLTP joins. No timestamp parsing, no multi-table chains, no string matching — just clean dimension filters.

In [ ]:
# --- Query 1: Revenue by Day of Week ---
print("=== Revenue by Day of Week ===")
print(f"{'day_name':<12} {'calls':>5} {'orders':>6} {'revenue':>10}")
print("-" * 36)
for r in cursor.execute('''
    SELECT dd.day_name, COUNT(*) AS calls,
           SUM(f.is_converted) AS orders,
           SUM(f.order_amount) AS revenue
    FROM fact_calls f
    JOIN dim_date dd ON f.date_key = dd.date_key
    GROUP BY dd.day_name
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<12} {r[1]:>5} {r[2]:>6}  ${r[3]:>8.2f}")

# --- Query 2: Conversion Rate by Time Period ---
print()
print("=== Conversion Rate by Time Period ===")
print(f"{'time_period':<12} {'calls':>5} {'conv':>5} {'rate':>7}")
print("-" * 32)
for r in cursor.execute('''
    SELECT dt.time_period, COUNT(*) AS calls,
           SUM(f.is_converted) AS conversions,
           ROUND(100.0 * SUM(f.is_converted) / COUNT(*), 1) AS conv_rate
    FROM fact_calls f
    JOIN dim_time dt ON f.time_key = dt.time_key
    GROUP BY dt.time_period
    ORDER BY conv_rate DESC
'''):
    print(f"{r[0]:<12} {r[1]:>5} {r[2]:>5} {r[3]:>6.1f}%")

# --- Query 3: Revenue by Product Category ---
print()
print("=== Revenue by Product Category ===")
print(f"{'category':<22} {'orders':>6} {'revenue':>10}")
print("-" * 40)
for r in cursor.execute('''
    SELECT dp.category,
           SUM(f.is_converted) AS orders,
           SUM(f.order_amount) AS revenue
    FROM fact_calls f
    JOIN dim_product dp ON f.product_key = dp.product_key
    WHERE dp.product_key > 0
    GROUP BY dp.category
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<22} {r[1]:>6}  ${r[2]:>8.2f}")

# --- Query 4: Campaign Performance Dashboard ---
print()
print("=== Campaign Performance Dashboard ===")
print(f"{'campaign':<24} {'client':<18} {'calls':>5} {'orders':>6} {'revenue':>10} {'conv%':>6}")
print("-" * 74)
for r in cursor.execute('''
    SELECT dcamp.campaign_name, dcamp.client_name,
           COUNT(*) AS calls,
           SUM(f.is_converted) AS orders,
           SUM(f.order_amount) AS revenue,
           ROUND(100.0 * SUM(f.is_converted) / COUNT(*), 1) AS conv_rate
    FROM fact_calls f
    JOIN dim_campaign dcamp ON f.campaign_key = dcamp.campaign_key
    GROUP BY dcamp.campaign_name, dcamp.client_name
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<24} {r[1]:<18} {r[2]:>5} {r[3]:>6}  ${r[4]:>8.2f} {r[5]:>5.1f}%")

print()
print("No complex string parsing, no timestamp math, no multi-table chains.")
print("The star schema pre-computed all of that.")

In [ ]:
# --- Same Question, Two Ways ---
# "Revenue by campaign by day of week (orders only)"

# OLTP version: 3 JOINs, CASE/strftime for day name, string-based GROUP BY
print("=== OLTP Query: Revenue by Campaign by Day ===")
print(f"{'campaign':<24} {'day':<11} {'orders':>6} {'revenue':>10}")
print("-" * 54)
for r in cursor.execute('''
    SELECT s.campaign_name,
           CASE strftime('%w', datetime(c.call_started_at, '-5 hours'))
               WHEN '0' THEN 'Sunday'    WHEN '1' THEN 'Monday'
               WHEN '2' THEN 'Tuesday'   WHEN '3' THEN 'Wednesday'
               WHEN '4' THEN 'Thursday'  WHEN '5' THEN 'Friday'
               WHEN '6' THEN 'Saturday'
           END AS day_name,
           COUNT(*) AS orders,
           SUM(o.total) AS revenue
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis
    JOIN orders o ON o.call_id = c.call_id
    GROUP BY s.campaign_name, day_name
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<24} {r[1]:<11} {r[2]:>6}  ${r[3]:>8.2f}")

# Star schema version: 2 JOINs, clean columns, no parsing
print()
print("=== Star Schema Query: Same Question ===")
print(f"{'campaign':<24} {'day':<11} {'orders':>6} {'revenue':>10}")
print("-" * 54)
for r in cursor.execute('''
    SELECT dc.campaign_name, dd.day_name,
           COUNT(*) AS orders,
           SUM(f.order_amount) AS revenue
    FROM fact_calls f
    JOIN dim_campaign dc ON f.campaign_key = dc.campaign_key
    JOIN dim_date dd ON f.date_key = dd.date_key
    WHERE f.is_converted = 1
    GROUP BY dc.campaign_name, dd.day_name
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<24} {r[1]:<11} {r[2]:>6}  ${r[3]:>8.2f}")

print()
print("Same answer. The star schema query is simpler, faster on large data,")
print("and impossible to get the timezone wrong — it was handled once during the ETL.")

---

## 7. Partitioning & Bucketing

**One-line answer:** Partitioning splits a table into folders by a column value. The query engine skips folders it doesn't need.

### Partitioning

Imagine `fact_calls` stored on disk. Without partitioning, every query reads every row. With partitioning by date:

```
fact_calls/
  date=2026-03-10/   ← 25 rows
  date=2026-03-11/   ← 15 rows
```

A query for March 10 only reads the first folder — the engine **prunes** the March 11 partition entirely.

### Bucketing (Clustering)

Bucketing hash-distributes rows within each partition. If you frequently join on `campaign_key`, bucketing by `campaign_key` puts related rows together on disk — faster joins.

| | Partitioning | Bucketing |
|---|-------------|-----------|
| **What** | Splits table into folders by column value | Hash-distributes rows within partitions |
| **When** | Filter columns (date, region, status) | Join/group columns (customer_key, campaign_key) |
| **Benefit** | Skip entire folders (partition pruning) | Co-locate related rows for faster joins |
| **Example** | `PARTITIONED BY (date)` | `CLUSTERED BY (campaign_key)` |


> **Analogy:** Think of partitions as filing cabinets (one per month) and bucketing as organizing the folders *inside* each cabinet by customer name — so when you need “all calls from Campaign A in March,” you go straight to the right drawer within the right cabinet.

> **In SQLite, there is no physical partitioning.** In BigQuery, Spark, and Hive, it is critical for cost and performance. You will implement it in M08 (BigQuery) and M09 (Delta Lake).

In [ ]:
# --- Partitioning Simulation ---

# Full table scan
full = cursor.execute("SELECT COUNT(*) FROM fact_calls").fetchone()[0]

# "Partitioned" query — only March 10
part = cursor.execute('''
    SELECT COUNT(*) FROM fact_calls f
    JOIN dim_date dd ON f.date_key = dd.date_key
    WHERE dd.full_date = '2026-03-10'
''').fetchone()[0]

print("=== Partition Pruning Simulation ===")
print(f"  Full table scan:       {full} rows read")
print(f"  Partitioned (Mar 10):  {part} rows read")
print(f"  Rows skipped:          {full - part} rows ({100*(full-part)/full:.1f}% fewer reads)")

# NOTE: BigQuery charges $5 per terabyte of data scanned per query — this is real
# production pricing. Partitioning reduces the data scanned, which directly reduces your bill.
# BigQuery cost projection
print()
print("=== BigQuery Cost Projection (1 TB table) ===")
full_cost = 5.00
part_cost = round(5.00 * part / full, 2)
print(f"  Without partitioning:  ${full_cost:.2f} per query")
print(f"  With date partition:   ${part_cost:.2f} per query")
print(f"  Savings per query:     ${full_cost - part_cost:.2f}")
print(f"  Daily savings (10 queries): ${(full_cost - part_cost) * 10:.2f}")

# Clustering simulation — sorting improves locality
print()
print("=== Clustering Simulation ===")
print("Rows sorted by campaign_key for faster GROUP BY:")
print(f"{'campaign_key':>12}  {'call_count':>10}")
print("-" * 26)
for r in cursor.execute('''
    SELECT campaign_key, COUNT(*) as cnt
    FROM fact_calls
    GROUP BY campaign_key
    ORDER BY campaign_key
'''):
    print(f"{r[0]:>12}  {r[1]:>10}")

print()
print("With clustering, these groups are physically adjacent on disk.")
print("You will implement real partitioning in M08 (BigQuery) and M09 (Delta Lake).")

---

## 8. Challenge Lab

Three challenges using the star schema you just built. Write each query, run it, and verify the results make sense.

In [ ]:
# TRY IT YOURSELF FIRST! Attempt the query before looking at the solution below.
# --- Challenge 1: Top 3 Products by Revenue ---
# JOIN fact_calls + dim_product, GROUP BY product_name, ORDER BY revenue DESC

# ============ SOLUTION ============
print("=== Challenge 1: Top 3 Products by Revenue ===")
print(f"{'product_name':<30} {'category':<20} {'orders':>6} {'revenue':>10}")
print("-" * 70)
for r in cursor.execute('''
    SELECT dp.product_name, dp.category,
           SUM(f.is_converted) AS orders,
           SUM(f.order_amount) AS revenue
    FROM fact_calls f
    JOIN dim_product dp ON f.product_key = dp.product_key
    WHERE dp.product_key > 0
    GROUP BY dp.product_name, dp.category
    ORDER BY revenue DESC
    LIMIT 3
'''):
    print(f"{r[0]:<30} {r[1]:<20} {r[2]:>6}  ${r[3]:>8.2f}")

In [ ]:
# TRY IT YOURSELF FIRST! Attempt the query before looking at the solution below.
# --- Challenge 2: Conversion Rate by Time Period ---
# Which time_period has the best conversion rate?

# ============ SOLUTION ============
print("=== Challenge 2: Best Time Period for Conversions ===")
print(f"{'time_period':<12} {'calls':>5} {'conversions':>11} {'conv_rate':>9}")
print("-" * 40)
for r in cursor.execute('''
    SELECT dt.time_period, COUNT(*) AS calls,
           SUM(f.is_converted) AS conversions,
           ROUND(100.0 * SUM(f.is_converted) / COUNT(*), 1) AS conv_rate
    FROM fact_calls f
    JOIN dim_time dt ON f.time_key = dt.time_key
    GROUP BY dt.time_period
    ORDER BY conv_rate DESC
'''):
    print(f"{r[0]:<12} {r[1]:>5} {r[2]:>11}    {r[3]:>5.1f}%")

In [ ]:
# TRY IT YOURSELF FIRST! Attempt the query before looking at the solution below.
# --- Challenge 3: SCD Type 2 Lookup ---
# For call_001 (caller 3135559876 on March 10), what city was the customer in?

# ============ SOLUTION ============
print("=== Challenge 3: Point-in-Time Customer Lookup ===")
print()

# Get the call details
call = cursor.execute('''
    SELECT call_id, caller_ani, DATE(call_started_at, '-5 hours') AS call_date
    FROM calls WHERE call_id = 'call_001'
''').fetchone()
print(f"Call: {call[0]}, Caller: {call[1]}, Date (EST): {call[2]}")

# Look up customer using SCD Type 2 date range
cust = cursor.execute('''
    SELECT dc.customer_key, dc.city, dc.state, dc.effective_date, dc.end_date
    FROM dim_customer dc
    WHERE dc.caller_ani = ?
      AND ? BETWEEN dc.effective_date AND dc.end_date
''', (call[1], call[2])).fetchone()

print(f"Customer key: {cust[0]}, City: {cust[1]}, State: {cust[2]}")
print(f"Version active: {cust[3]} to {cust[4]}")

# Verify in fact_calls
fact = cursor.execute('''
    SELECT f.call_id, f.customer_key, dc.city, dc.state
    FROM fact_calls f
    JOIN dim_customer dc ON f.customer_key = dc.customer_key
    WHERE f.call_id = 'call_001'
''').fetchone()
print()
print(f"Fact table confirms: {fact[0]} -> customer_key={fact[1]} -> {fact[2]}, {fact[3]}")
print()
print("The fact table used the SCD Type 2 date range to pick the correct customer version.")
print("On March 10, this caller was in Detroit. After March 11, they are in Chicago.")

---

## 9. Key Takeaways

1. **OLTP = operations, OLAP = analytics.** Star schema is the bridge between them.
2. **Grain statement first** — everything follows from "one row = one ___." Our grain: one row = one phone call.
3. **Fact tables hold measures (numbers).** Dimension tables hold context (who, what, when, where).
4. **Surrogate keys (integers) replace natural keys (strings)** — faster joins, handles changes gracefully.
5. **Default dimension row (key = 0) eliminates NULLs** in fact table foreign keys.
6. **SCD Type 2 preserves history** — same entity, multiple rows with effective/end date ranges.
7. **Point-in-time lookup:** `WHERE date BETWEEN effective_date AND end_date` — answers "what was true THEN?"
8. **Denormalize dimensions** — flatten normalized OLTP tables (sources + clients → dim_campaign) for query simplicity.
9. **Star schema queries are simpler and faster** than equivalent OLTP queries. The ETL handles complexity once.
10. **Partitioning = physical folder structure.** The query engine skips irrelevant partitions (partition pruning).
11. **This star schema IS the Gold layer** — you will build the Bronze → Silver → Gold pipeline in M06–M16.

---

## 10. Homework & Preview

### 1. Draw the Star Schema (15 min)

On paper or in [draw.io](https://draw.io), sketch the star schema:
- `fact_calls` in the center
- 5 dimension tables around it
- Label: primary keys, foreign keys, measures, and the grain statement

### 2. SCD Type 2 Scenario (15 min)

Acme Corp changes its name to "Acme Industries" on March 15. How would you handle this in `dim_campaign`?

Write the SQL to:
1. Close the old rows (set `end_date`, `is_current = 0`)
2. Insert new rows with the updated name

*Hint: You would need to add `effective_date`, `end_date`, and `is_current` columns to `dim_campaign` first.*

### 3. Cross-Domain Design Exercise (20 min)

Design a star schema for a **hospital emergency room**:
- **Fact table:** `fact_visits` (one row = one ER visit)
- What are the **dimensions**? (who, what, when, where, how)
- What are the **measures**? (numbers you would aggregate)
- What is the **grain statement**?
- Which dimension would benefit from **SCD Type 2**? Why?

### 4. Preview M05: Hadoop & Hive

Skim these concepts — 10 minutes of reading:
- "What is Hadoop?" — distributed storage + compute
- "What is Hive?" — SQL on Hadoop

We are moving from single-machine SQL (SQLite) to **distributed data processing** across clusters of machines.

---

**Next session:** M05 — Hadoop & Hive (distributed storage, MapReduce, HiveQL)